# Phase 2 — LaBSE Cross-Lingual Event Deduplication

**목적**: `event_dedup_input.parquet` (44,663 rows, 6 lang) 의 title을
LaBSE 로 임베딩하고, 시간 window ±3일 + 동일 company_id 블록 내에서
cosine ≥ τ 인 기사들을 Union-Find 로 묶어 `event_cluster_id` 를 부여한다.

**출력**: `event_clusters.parquet` (event_id, event_cluster_id, cluster_size,
is_cluster_lead, cluster_lead_tier)

**근거 논문**:
- Feng, Yang, Cer, Arivazhagan, Wang (2020/2022) *"Language-Agnostic BERT
  Sentence Embedding"* — LaBSE 원논문, 109개 언어 cross-lingual alignment.
- Rupnik et al. (2016) *"News across languages - Cross-Lingual Document
  Similarity and Event Tracking"* JAIR — 뉴스 이벤트 dedup window.
- Boschee et al. (2015) ICEWS, Leetaru & Schrodt (2013) GDELT — event
  deduplication in multilingual news streams.

**파라미터 (기본값은 Rupnik et al. 2016 + LaBSE 논문 경험치의 절충)**:
- `TIME_WINDOW_DAYS = 3` (±3일)
- `SIM_THRESHOLD = 0.80` (LaBSE parallel mining 권장 0.60~0.70 보다 엄격하게,
  '동일 이벤트 보도'만 묶기 위해 0.80 로 설정 — **추론**)
- blocking key: `company_id` (같은 기업에 대한 보도만 묶음)

In [ ]:
# 0) Google Drive 마운트
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT = '/content/drive/MyDrive/butterfly_effect'
assert os.path.exists(PROJECT), f'프로젝트 폴더 없음: {PROJECT}'
os.chdir(PROJECT)
print('cwd:', os.getcwd())

In [ ]:
# 1) 의존성 설치 (LaBSE + FAISS)
!pip install -q sentence-transformers faiss-cpu pyarrow tqdm

In [ ]:
# 2) GPU 확인 & 입력 로딩
import torch, pandas as pd, numpy as np
print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

INPUT = 'data/processed/event_dedup_input.parquet'
df = pd.read_parquet(INPUT)
print('rows:', len(df))
print('cols:', list(df.columns))
df.head(3)

In [ ]:
# 3) LaBSE 모델 로딩
from sentence_transformers import SentenceTransformer
model = SentenceTransformer('sentence-transformers/LaBSE')
if torch.cuda.is_available():
    model = model.to('cuda')
print('LaBSE loaded. dim =', model.get_sentence_embedding_dimension())

In [ ]:
# 4) 임베딩 생성 (정규화 포함 → inner product = cosine)
from tqdm.auto import tqdm
titles = df['title'].tolist()
emb = model.encode(
    titles,
    batch_size=256 if torch.cuda.is_available() else 64,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,  # L2-normalize
)
print('embeddings shape:', emb.shape, 'dtype:', emb.dtype)
np.save('data/processed/labse_embeddings.npy', emb.astype('float32'))
print('saved → data/processed/labse_embeddings.npy')

In [ ]:
# 5) Blocking: (company_id, date) 기반으로 후보쌍만 생성
#    time window ±3일 내에 같은 company_id 를 가진 pair 만 비교
import pandas as pd
from datetime import timedelta
TIME_WINDOW_DAYS = 3
SIM_THRESHOLD = 0.80

df['event_time'] = pd.to_datetime(df['event_time'])
df['_date'] = df['event_time'].dt.normalize()
df = df.sort_values(['company_id', '_date']).reset_index(drop=True)
# row index column for embedding lookup
df['_ridx'] = np.arange(len(df))
print('sorted df ready:', df.shape)

In [ ]:
# 6) company_id 별로 FAISS IndexFlatIP 로 후보쌍 생성
#    각 company 내에서 BFS window ±3일 비교
import faiss
from collections import defaultdict

edges = []  # (i_ridx, j_ridx, sim)
grouped = df.groupby('company_id')
total_pairs = 0
total_edges = 0

for cid, grp in tqdm(grouped, total=grouped.ngroups):
    if len(grp) < 2:
        continue
    ridx = grp['_ridx'].to_numpy()
    sub_emb = emb[ridx]
    dates = grp['_date'].to_numpy()

    # FAISS IP index on this company
    index = faiss.IndexFlatIP(sub_emb.shape[1])
    index.add(sub_emb.astype('float32'))
    # kNN with k = min(50, len)
    k = min(50, len(grp))
    sims, nbrs = index.search(sub_emb.astype('float32'), k)

    for i in range(len(grp)):
        for rank in range(1, k):  # skip self
            j = nbrs[i, rank]
            s = sims[i, rank]
            if s < SIM_THRESHOLD:
                break  # sorted desc
            if j <= i:
                continue
            # time window check
            dt = abs((dates[i] - dates[j]) / np.timedelta64(1, 'D'))
            if dt > TIME_WINDOW_DAYS:
                continue
            edges.append((int(ridx[i]), int(ridx[j]), float(s)))
            total_edges += 1
    total_pairs += len(grp) * (len(grp) - 1) // 2

print(f'candidate pairs (blocked): {total_pairs:,}')
print(f'edges (sim ≥ {SIM_THRESHOLD}, |Δt|≤{TIME_WINDOW_DAYS}d): {total_edges:,}')

In [ ]:
# 7) Union-Find 로 connected components → event_cluster_id
class DSU:
    def __init__(self, n):
        self.p = list(range(n))
        self.r = [0]*n
    def find(self, x):
        while self.p[x] != x:
            self.p[x] = self.p[self.p[x]]
            x = self.p[x]
        return x
    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra == rb: return
        if self.r[ra] < self.r[rb]: ra, rb = rb, ra
        self.p[rb] = ra
        if self.r[ra] == self.r[rb]: self.r[ra] += 1

dsu = DSU(len(df))
for i, j, _ in edges:
    dsu.union(i, j)

cluster_root = np.array([dsu.find(i) for i in range(len(df))])
# map root → contiguous cluster id
unique_roots, inv = np.unique(cluster_root, return_inverse=True)
df['event_cluster_id'] = inv
n_clusters = len(unique_roots)
n_multi = (pd.Series(inv).value_counts() > 1).sum()
print(f'total clusters: {n_clusters:,}')
print(f'multi-article clusters: {n_multi:,}')
print(f'singletons: {n_clusters - n_multi:,}')
print(f'dedup ratio: {len(df)/n_clusters:.3f}x')

In [ ]:
# 8) cluster_size, is_cluster_lead, cluster_lead_tier 계산
#    lead = (source_tier 오름차순, event_time 오름차순) 첫 행
df['cluster_size'] = df.groupby('event_cluster_id')['event_cluster_id'].transform('size')

df_sorted = df.sort_values(['event_cluster_id', 'source_tier', 'event_time', 'event_id'])
lead_idx = df_sorted.groupby('event_cluster_id').head(1).index
df['is_cluster_lead'] = False
df.loc[lead_idx, 'is_cluster_lead'] = True

df['cluster_lead_tier'] = df.groupby('event_cluster_id')['source_tier'].transform('min')

print('lead articles:', df['is_cluster_lead'].sum())
print('cluster size distribution:')
print(df['cluster_size'].describe())
print('\ncluster_lead_tier 분포:')
print(df['cluster_lead_tier'].value_counts().sort_index())

In [ ]:
# 9) 검증: 다국어 cluster 샘플 (same cluster 내 서로 다른 finbert_model)
multi_lang = (df.groupby('event_cluster_id')['finbert_model'].nunique() >= 2)
multi_lang_clusters = multi_lang[multi_lang].index[:10]
for cid in multi_lang_clusters:
    sub = df[df['event_cluster_id'] == cid][['event_time','finbert_model','source_tier','publisher','title']]
    print(f'\n=== cluster {cid} (size={len(sub)}) ===')
    for _, r in sub.iterrows():
        print(f"  [{r.finbert_model[:6]}] T{int(r.source_tier)} {r.event_time.date()}  {str(r.publisher)[:20]:<20} | {r.title[:80]}")

In [ ]:
# 10) 저장
out_cols = ['event_id', 'event_cluster_id', 'cluster_size',
            'is_cluster_lead', 'cluster_lead_tier']
out_df = df[out_cols].copy()
OUT = 'data/processed/event_clusters.parquet'
out_df.to_parquet(OUT, index=False)
print(f'wrote {OUT}  ({len(out_df):,} rows)')
print(out_df.head())

## 완료 — VM 으로 복사해야 할 파일

- `data/processed/event_clusters.parquet` (필수)
- `data/processed/labse_embeddings.npy` (선택, 후속 재사용용)

그 다음 VM 에서 `python scripts/merge_event_clusters_to_v9.py` 실행.